# 08 · Probabilistic shaping and coded modulation

A shaped constellation (01 · section 5) sends inner points more often.
This notebook covers how the transmitter actually produces such symbols
from bits, and how shaping combines with a FEC code:

1. **distribution matching**: turning uniform bits into shaped symbols,
   invertibly (CCDM), and its rate loss;
2. the shaping gain in GMI against uniform QAM;
3. **probabilistic amplitude shaping** (PAS): a matcher and a FEC code
   together, compared with uniform QAM at the same net rate.

It builds on [01 · Constellations and bit mapping](01-constellations-and-bit-mapping.ipynb),
[06 · AWGN and performance metrics](06-awgn-and-performance-metrics.ipynb) and
[07 · Channel coding](07-channel-coding.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import Constellation
from commkit.backend import to_device
from commkit.coding import CCDM, PAS, LDPCCode, SumProduct, decode

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()


def host(a):
    return to_device(a, "cpu")

## 1. Distribution matching

A constant-composition distribution matcher (CCDM) maps `k` uniform bits
to a block of `n` points that holds each point a fixed number of times,
the composition closest to the target pmf.  Different inputs give
different arrangements, and the dematcher inverts them exactly.  A shaped
`generate` runs one matcher block per channel.  Its reference keeps the
labels on the air as `bits` and the matcher's input as `info_bits`:

In [ ]:
ps = Constellation.qam(64).shaped(entropy=5.0)
tx = ck.generate(ps, 20000, symbol_rate=1e9, rng=1, device=DEVICE)
ref = tx.reference
print(
    "labels on the air:",
    ref.bits.shape[-1],
    "bits;  source bits:",
    ref.info_bits.shape[-1],
)
print(
    f"rate {ref.info_bits.shape[-1] / 20000:.4f} bit/symbol, entropy {ps.entropy:.4f}"
)
counts = np.bincount(host(ps._nearest(ref.symbols)), minlength=64)
matcher = CCDM(ps, block_length=20000)
print(
    "point counts equal the composition:", np.array_equal(counts, matcher.composition)
)
print(
    "dematched bits equal the source:",
    np.array_equal(host(decode(ref.bits, code=matcher).bits), host(ref.info_bits)),
)

A fixed composition costs rate: there are fewer sequences of one
composition than i.i.d. sequences of the same entropy.  The loss falls
like `log(n) / n`, so long blocks are nearly free:

In [ ]:
for n in (64, 256, 1024, 4096, 65536):
    m = CCDM(ps, block_length=n)
    print(f"n = {n:6d}: {m.k / n:.4f} bit/symbol, loss {ps.entropy - m.k / n:.4f}")

## 2. The shaping gain

At equal average power, the shaped constellation's GMI exceeds the
uniform one's at intermediate SNR: part of the 1.53 dB gap between square
QAM and the Shannon capacity is recovered.  At high SNR the entropy caps
it: an entropy of 5 bits cannot carry more than 5 bit/symbol.

In [ ]:
grid = np.arange(6, 21, 2.0)
formats = {
    "16-QAM": Constellation.qam(16),
    "64-QAM": Constellation.qam(64),
    "PS-64-QAM, H = 5": ps,
}
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(grid, np.log2(1 + 10 ** (grid / 10)), "--", color="gray", label="capacity")
gmi = {}
for name, c in formats.items():
    sig = ck.generate(c, 2**14, symbol_rate=1e9, rng=2, device=DEVICE)
    gmi[name] = [
        ck.metrics.gmi(
            ck.impairments.apply_awgn(sig, esn0_db=e, rng=3), noise_var=10 ** (-e / 10)
        )
        for e in grid
    ]
    ax.plot(grid, gmi[name], "o-", label=name)
ax.set_xlabel("Es/N0 [dB]")
ax.set_ylabel("GMI [bit/symbol]")
ax.legend()
plt.show()
for e, a, b in zip(grid, gmi["64-QAM"], gmi["PS-64-QAM, H = 5"], strict=True):
    print(f"{e:4.1f} dB: 64-QAM {a:.3f}, PS-64-QAM {b:.3f} bit/symbol")

The shaped constellation carries about 0.3 bit/symbol more up to 12 dB,
and the uniform one wins beyond about 15 dB, as the entropy cap predicts.

## 3. Probabilistic amplitude shaping

A FEC code cannot simply follow the matcher: its parity bits are uniform
and would undo the shaping.  PAS keeps them where uniform bits belong.
In Gray QAM each real dimension has one **sign** bit (the MSB) and `m - 1`
**amplitude** bits, and `±a` share the amplitude label.  So:

1. the matcher produces the amplitudes, shaped;
2. a systematic FEC encodes the amplitude bits (and a few uniform bits);
3. its parity bits become the signs, which are uniform anyway.

The code rate must be at least `(m - 1) / m`.  For 64-QAM with the DVB-S2
short-frame rate-11/15 code (0.733), the net rate is
`H - 6 (1 - R_c)` bit/symbol.  To compare with uniform 16-QAM and the
rate-8/9 code (3.556 bit/symbol), choose the entropy that gives the same
net rate:

In [ ]:
fec = LDPCCode.dvbs2("3/4", "short")  # k = 11880 of 16200: rate 11/15
uniform_code = LDPCCode.dvbs2("8/9", "short")
target = uniform_code.rate * 4
shaped = Constellation.qam(64).shaped(entropy=target + 6 * (1 - fec.rate))
pas = PAS(fec, shaped)
print(f"uniform 16-QAM + rate 8/9: {target:.3f} bit/symbol")
print(
    f"PAS, PS-64-QAM H = {shaped.entropy:.3f} + rate 11/15: {pas.k / (pas.n / 6):.3f} bit/symbol"
)

In [ ]:
links = {
    "PAS, PS-64-QAM": (shaped, pas),
    "16-QAM, uniform": (Constellation.qam(16), uniform_code),
}
grid = np.arange(10.5, 13.01, 0.5)
fer = {name: [] for name in links}
for name, (c, code) in links.items():
    tx = ck.generate(c, symbol_rate=1e9, code=code, num_blocks=12, rng=4, device=DEVICE)
    for e in grid:
        rx = ck.impairments.apply_awgn(tx, esn0_db=e, rng=5)
        llrs = ck.mapping.compute_llr(
            rx, noise_var=10 ** (-e / 10)
        )  # includes the prior
        res = decode(llrs, code=code, method=SumProduct())
        fer[name].append(
            ck.metrics.fer(res.bits, tx.reference.info_bits, block_length=code.k)
        )
for i, e in enumerate(grid):
    print(f"Es/N0 {e:4.1f} dB:", ", ".join(f"{n} FER {fer[n][i]:.2f}" for n in links))

At the same net rate, PAS decodes from 11.5 dB and uniform 16-QAM only from
about 12.5 dB: about 1 dB of shaping gain, realized with a real code.  The LLRs fed to
the decoder include the prior of the shaped constellation (`compute_llr`
does that by itself), and the GMI that predicts decoding is that of the
shaped labels.

**Summary**

* Shaped symbols come from bits through a distribution matcher; `generate`
  keeps the matcher's input as `info_bits`.
* A constant composition costs a rate loss that vanishes as `log(n) / n`.
* PAS lets a systematic FEC code and a matcher work together, the parity
  bits becoming the signs.

**Next:** [09 · Laser phase noise](09-laser-phase-noise.ipynb).